# 0. ML Stock Lab Experiments

Notebook-first lab for agnostic fundamental valuation, ML mispricing signals and quintile portfolio evaluation. Uses `ml_stock_lab` and the existing Drive-first data platform.


In [1]:
# 1. Setup / Parameters
from pathlib import Path
import sys

START_DIR = Path.cwd()
PACKAGE_ROOT = START_DIR
PROJECT_ROOT = START_DIR

for parent in [START_DIR, *START_DIR.parents]:
    if (parent / "ml_stock_lab").exists():
        PACKAGE_ROOT = parent
        break

for parent in [START_DIR, *START_DIR.parents]:
    if (parent / "notebooks" / "03_ML_Equity_Stock_Lab.ipynb").exists() or (parent / "machine_learning_lab" / "notebooks" / "ML_Stock_Lab_Experiments.ipynb").exists():
        PROJECT_ROOT = parent
        break
    if (parent / "research_platform_definitive" / "notebooks" / "03_ML_Equity_Stock_Lab.ipynb").exists():
        PROJECT_ROOT = parent / "research_platform_definitive"
        break

for candidate in [PACKAGE_ROOT, PROJECT_ROOT]:
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

OUTPUTROOT = PROJECT_ROOT / "output" / "ml_stock_lab"
TABLESDIR = OUTPUTROOT / "tables"
FIGURESDIR = OUTPUTROOT / "figures"
TABLESDIR.mkdir(parents=True, exist_ok=True)
FIGURESDIR.mkdir(parents=True, exist_ok=True)
MODEL = "ols"  # ols, lasso, rf, gbrt, ensemble
MAX_ROWS = 2000
MIN_DATES = 2
MIN_TICKERS = 5
print("PACKAGE_ROOT", PACKAGE_ROOT)
print("PROJECT_ROOT", PROJECT_ROOT)
print("OUTPUTROOT", OUTPUTROOT)


PACKAGE_ROOT /Users/itsgennymac/GitHub/machine-learning-for-trading
PROJECT_ROOT /Users/itsgennymac/GitHub/machine-learning-for-trading/research_platform_definitive
OUTPUTROOT /Users/itsgennymac/GitHub/machine-learning-for-trading/research_platform_definitive/output/ml_stock_lab


## 2. Imports


In [2]:
import pandas as pd
import plotly.express as px

# Se stai lavorando in Colab, monta il repo locale o usa `pip install -e .`.
# Qui assumiamo che `ml_stock_lab` sia gia importabile dal repo montato.
from ml_stock_lab.datasets.panel import (
    FundamentalDatasetBuilder,
    add_basic_features,
    load_financial_db_panel,
    make_forward_returns,
    select_numeric_features,
)
from ml_stock_lab.valuation.peer_ols import PeerImpliedValuator
from ml_stock_lab.signals.mispricing import compute_absolute_mispricing, compute_relative_mispricing, cross_sectional_zscore
from ml_stock_lab.signals.quintiles import make_quantile_portfolios, rank_scores
from ml_stock_lab.prediction.expected_returns import (
    ExpectedReturnModel,
    describe_temporal_split,
    oos_r2,
    temporal_train_test_split,
)
from ml_stock_lab.evaluation.diagnostics import (
    ExperimentStatus,
    build_status_from_signals,
    save_status_file,
    should_run_experiment,
    validate_panel,
)
from ml_stock_lab.evaluation.performance import annualized_volatility, quintile_risk_report, sharpe_ratio
from ml_stock_lab.tracking.summary import build_experiment_summary, save_experiment_summary


## 2.5 Fintech ML Lab Control Center


In [3]:
# 2.5 Fintech ML Lab Control Center - Universe, Model, Currency and APIs
import os
from IPython.display import display, clear_output, HTML
try:
    import ipywidgets as widgets
    _WIDGETS_OK = True
except Exception:
    widgets = None
    _WIDGETS_OK = False

ML_LAB_UNIVERSES = {
    "US Mega Cap": ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META", "LLY", "JPM"],
    "Italy Banks": ["ISP.MI", "UCG.MI", "BAMI.MI", "BMED.MI", "MB.MI"],
    "Europe Quality": ["ASML.AS", "SAP.DE", "NESN.SW", "NOVO-B.CO", "RMS.PA", "MC.PA"],
    "Semiconductors": ["NVDA", "AMD", "AVGO", "QCOM", "INTC", "TSM", "ASML"],
    "All Available": [],
    "Manual": [],
}
API_KEY_FIELDS = {"FMP_API_KEY": "FMP", "FINNHUB_API_KEY": "Finnhub", "ALPHA_VANTAGE_API_KEY": "Alpha Vantage", "EODHD_API_KEY": "EODHD", "FRED_API_KEY": "FRED"}
CURRENCY_FX = {"USD": "DX-Y.NYB", "EUR": "EURUSD=X", "GBP": "GBPUSD=X", "CHF": "CHF=X", "JPY": "JPY=X"}
ML_LAB_CONFIG = globals().get("ML_LAB_CONFIG", {})

def _parse_tickers(text):
    out=[]
    for item in str(text or "").replace("\n", ",").replace(";", ",").split(","):
        t=item.strip().upper()
        if t and t not in out: out.append(t)
    return out

def _parse_keys(text):
    out={}
    for line in str(text or "").splitlines():
        line=line.strip()
        if not line or line.startswith("#"): continue
        sep="=" if "=" in line else ":" if ":" in line else None
        if sep:
            k,v=line.split(sep,1); k=k.strip().upper(); v=v.strip().strip('"').strip("'")
            if k in API_KEY_FIELDS and v: out[k]=v
    return out

def _sync_ml_lab_config(cfg):
    global MODEL, MAX_ROWS, MIN_DATES, MIN_TICKERS, ML_LAB_CONFIG
    MODEL = cfg["model"]
    MAX_ROWS = int(cfg["max_rows"])
    MIN_DATES = int(cfg.get("min_dates", MIN_DATES))
    MIN_TICKERS = int(cfg.get("min_tickers", MIN_TICKERS))
    ML_LAB_CONFIG = cfg

_default = {
    "model": globals().get("MODEL", "ols"),
    "max_rows": int(globals().get("MAX_ROWS", 2000)),
    "min_dates": int(globals().get("MIN_DATES", 2)),
    "min_tickers": int(globals().get("MIN_TICKERS", 5)),
    "universe": "All Available",
    "tickers": [],
    "currency": "USD",
    "fx_pair": "DX-Y.NYB",
    "feature_blocks": ["value", "quality", "momentum"],
    "target": "market_value",
    "refresh_cache": False,
}
_sync_ml_lab_config({**_default, **ML_LAB_CONFIG})

if not _WIDGETS_OK:
    display(HTML("<div style='background:#fff7ed;border-left:5px solid #da7101;padding:12px;border-radius:8px'>ipywidgets unavailable. Using ML_LAB_CONFIG defaults.</div>"))
else:
    STYLE={"description_width":"120px"}
    universe_w=widgets.Dropdown(options=list(ML_LAB_UNIVERSES.keys()), value=ML_LAB_CONFIG.get("universe", "All Available"), description="Universe", style=STYLE, layout=widgets.Layout(width="310px"))
    tickers_w=widgets.Textarea(value=", ".join(ML_LAB_CONFIG.get("tickers", [])), description="Tickers", style=STYLE, layout=widgets.Layout(width="720px", height="80px"))
    model_w=widgets.Dropdown(options=["ols", "lasso", "rf", "gbrt", "ensemble"], value=ML_LAB_CONFIG.get("model", MODEL), description="Model", style=STYLE, layout=widgets.Layout(width="260px"))
    max_rows_w=widgets.IntSlider(value=int(ML_LAB_CONFIG.get("max_rows", MAX_ROWS)), min=100, max=20000, step=100, description="Max rows", style=STYLE, layout=widgets.Layout(width="420px"))
    target_w=widgets.Dropdown(options=["market_value", "forward_return", "mispricing_rel"], value=ML_LAB_CONFIG.get("target", "market_value"), description="Target", style=STYLE, layout=widgets.Layout(width="260px"))
    features_w=widgets.SelectMultiple(options=["value", "quality", "growth", "momentum", "risk", "macro_fx"], value=tuple(ML_LAB_CONFIG.get("feature_blocks", ["value", "quality", "momentum"])), description="Features", style=STYLE, layout=widgets.Layout(width="430px", height="120px"))
    currency_w=widgets.Dropdown(options=list(CURRENCY_FX.keys()), value=ML_LAB_CONFIG.get("currency", "USD"), description="Currency", style=STYLE, layout=widgets.Layout(width="260px"))
    fx_w=widgets.Text(value=ML_LAB_CONFIG.get("fx_pair", "DX-Y.NYB"), description="FX proxy", style=STYLE, layout=widgets.Layout(width="260px"))
    refresh_w=widgets.Checkbox(value=bool(ML_LAB_CONFIG.get("refresh_cache", False)), description="Refresh stale cache/API when available", indent=False)
    api_text=widgets.Textarea(value="", placeholder="FMP_API_KEY=...\nFRED_API_KEY=...", description="API keys", style=STYLE, layout=widgets.Layout(width="720px", height="80px"))
    apply_btn=widgets.Button(description="Apply ML Lab setup", icon="check", button_style="success", layout=widgets.Layout(width="190px", height="40px"))
    out=widgets.Output()

    def _universe_changed(change=None):
        values=ML_LAB_UNIVERSES.get(universe_w.value, [])
        tickers_w.value=", ".join(values)
    def _currency_changed(change=None):
        fx_w.value=CURRENCY_FX.get(currency_w.value, fx_w.value)
    def _apply(_=None):
        with out:
            clear_output(wait=True)
            for k,v in _parse_keys(api_text.value).items(): os.environ[k]=v
            api_text.value=""
            cfg={"model": model_w.value, "max_rows": int(max_rows_w.value), "min_dates": MIN_DATES, "min_tickers": MIN_TICKERS, "universe": universe_w.value, "tickers": _parse_tickers(tickers_w.value), "currency": currency_w.value, "fx_pair": fx_w.value.strip(), "feature_blocks": list(features_w.value), "target": target_w.value, "refresh_cache": bool(refresh_w.value)}
            _sync_ml_lab_config(cfg)
            display(HTML(f"<div style='background:#f6f8fb;border-left:5px solid #01696f;border-radius:8px;padding:12px'><b>ML Lab setup applied.</b> Model {MODEL}, rows {MAX_ROWS}, universe {cfg['universe']} ({len(cfg['tickers']) or 'all'} tickers), min coverage {MIN_DATES} dates / {MIN_TICKERS} tickers.</div>"))
    universe_w.observe(_universe_changed, names="value")
    currency_w.observe(_currency_changed, names="value")
    apply_btn.on_click(_apply)
    display(HTML("""
    <div style='background:#f6f8fb;border:1px solid #d9e2ec;border-left:5px solid #01696f;border-radius:10px;padding:14px;margin:10px 0'>
      <h3 style='margin:0;color:#01696f'>ML Equity Lab Control Center</h3>
      <p style='margin:6px 0 0;color:#344054'>Choose universe, model, target, feature blocks, reporting currency and FX proxy before the panel is loaded.</p>
    </div>
    """))
    display(widgets.VBox([widgets.HBox([universe_w, model_w, target_w]), tickers_w, widgets.HBox([features_w, max_rows_w]), widgets.HBox([currency_w, fx_w, refresh_w]), api_text, apply_btn, out]))
    _apply()


## 3. Data Platform Bootstrap

Drive-first / cache-second / API-last. The lab first reuses existing project artifacts and Database Finanziario-compatible panels.


In [4]:
panel = load_financial_db_panel(output_root=PROJECT_ROOT)
requested_tickers = set(ML_LAB_CONFIG.get("tickers", []))
if requested_tickers and not panel.empty and "ticker" in panel.columns:
    filtered_panel = panel[panel["ticker"].astype(str).str.upper().isin(requested_tickers)].copy()
    if not filtered_panel.empty:
        panel = filtered_panel
panel = add_basic_features(panel).head(MAX_ROWS)
if "forward_return" not in panel.columns:
    panel = make_forward_returns(panel, price_col="price" if "price" in panel.columns else "market_value")

panel_val = validate_panel(panel, min_dates=MIN_DATES, min_tickers=MIN_TICKERS)
panel_should_run = should_run_experiment(panel, min_dates=MIN_DATES, min_tickers=MIN_TICKERS)
if not panel_val.ok:
    print(f"[WARN] Panel validation failed: {panel_val.reason}")
    experiment_status = ExperimentStatus(
        status="error",
        reason="panel_validation_failed",
        details={
            "min_date": panel_val.min_date,
            "max_date": panel_val.max_date,
            "n_dates": panel_val.n_dates,
            "n_tickers": panel_val.n_tickers,
        },
    )
    save_status_file(experiment_status, OUTPUTROOT)
else:
    experiment_status = ExperimentStatus(status="ok", details={"stage": "panel_validation"})

panel.to_csv(TABLESDIR / "MLStockLab_panel.csv", index=False)
print("Panel shape:", panel.shape)
print("Panel validation:", panel_val)
display(panel.head())


Panel shape: (151, 40)
Panel validation: PanelValidationResult(ok=True, min_date='2024-03-31', max_date='2026-05-21', n_dates=3, n_tickers=76, reason=None, extra={'min_dates': 2, 'min_tickers': 5})


,screener_rank,ticker,company_name,country,fundamental_matched,robustness_status,scenario_downside,screener_score,universe_source,updated_at,...,portfolio,prediction,actual,N,prediction_rank,selected_topk,source,log_market_value,target_source,forward_return
0,NaN,RETAIL_MORTGAGES,Retail_Mortgages,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-20T15:01:05+00:00,...,Retail_Mortgages,336.9133,252.2316,4.0,1.0,1.0,workspace_backtest_topk,5.530348,actual,NaN
1,4.0,A2A.MI,NaN,Italy,NaN,WARN,NaN,0.0,static_ftse_mib,2026-05-20T01:50:41+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,actual,NaN
2,5.0,AAPL,NaN,NaN,NaN,WARN,NaN,0.0,static_sp500_core_sample,2026-05-20T01:50:41+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,actual,NaN
3,6.0,ABBV,NaN,NaN,NaN,WARN,NaN,0.0,static_sp500_core_sample,2026-05-20T01:50:41+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,actual,NaN
4,7.0,AI.PA,NaN,France,NaN,WARN,NaN,0.0,static_euro_stoxx_50,2026-05-20T01:50:41+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,actual,NaN


## 4. Feature Engineering


In [5]:
if not panel_should_run:
    feature_cols = []
    X = pd.DataFrame()
    y = pd.Series(dtype=float)
    print("Skipping feature build because panel validation did not pass should-run checks.")
else:
    feature_cols = select_numeric_features(panel, target="market_value", min_non_null=max(3, min(10, len(panel)//10)))
    if not feature_cols:
        feature_cols = [c for c in panel.select_dtypes("number").columns if c not in {"market_value", "forward_return"}][:8]
    print("Features:", feature_cols)
    X, y, feature_cols = FundamentalDatasetBuilder(feature_cols, "market_value").build(panel)
    print(X.shape, y.shape)


Features: ['screener_rank', 'screener_score', 'source_count', 'is_etf', 'is_adr']
(75, 5) (75,)


## 5. Agnostic Fundamental Analysis

Fair value: $\hat{V}_{i,t}=f_t(X_{i,t})$. Mispricing: $(\hat{V}-V^{mkt})/V^{mkt}$. Cross-sectional z-score by date.


In [6]:
if X.empty or not feature_cols:
    signals = pd.DataFrame()
    signals.to_csv(TABLESDIR / "MLStockLab_signals.csv", index=False)
    experiment_status = ExperimentStatus(status="error", reason="signals_empty", details={"stage": "fair_value", "panel_rows": len(panel)})
    save_status_file(experiment_status, OUTPUTROOT)
    print("[WARN] Signals skipped: no trainable feature matrix.")
else:
    try:
        valuator = PeerImpliedValuator(model=MODEL)
        fair_value = valuator.fit_predict(X, y)
        signals = panel.loc[X.index].copy()
        signals["fair_value_hat"] = fair_value
        signals["mispricing_abs"] = compute_absolute_mispricing(signals["fair_value_hat"], signals["market_value"])
        signals["mispricing_rel"] = compute_relative_mispricing(signals["fair_value_hat"], signals["market_value"])
        zscore_source_col = "mispricing_rel"
        if pd.to_numeric(signals["mispricing_rel"], errors="coerce").notna().sum() < max(MIN_TICKERS, 10):
            zscore_source_col = "mispricing_abs"
            print("[WARN] mispricing_rel unavailable or too sparse; using mispricing_abs for signal z-score fallback.")
        signals["zscore"] = cross_sectional_zscore(signals[zscore_source_col], signals["date"] if "date" in signals.columns else None)
        if signals["zscore"].notna().sum() < MIN_TICKERS:
            print("[WARN] cross-sectional z-score sparse; using global z-score fallback.")
            signals["zscore"] = cross_sectional_zscore(signals[zscore_source_col], None)
        signals["zscore_source"] = zscore_source_col
        signals = rank_scores(signals, "zscore", ascending=False)
    except Exception as exc:
        signals = pd.DataFrame()
        experiment_status = ExperimentStatus(status="error", reason="signals_empty", details={"stage": "fair_value", "error": type(exc).__name__})
        save_status_file(experiment_status, OUTPUTROOT)
        print(f"[WARN] Signal generation failed: {type(exc).__name__}: {exc}")
    signals.to_csv(TABLESDIR / "MLStockLab_signals.csv", index=False)
    display(signals.head(25))


[WARN] mispricing_rel unavailable or too sparse; using mispricing_abs for signal z-score fallback.


,screener_rank,ticker,company_name,country,fundamental_matched,robustness_status,scenario_downside,screener_score,universe_source,updated_at,...,source,log_market_value,target_source,forward_return,fair_value_hat,mispricing_abs,mispricing_rel,zscore,zscore_source,percentile_rank
1,4.0,A2A.MI,NaN,Italy,NaN,WARN,NaN,0.0,static_ftse_mib,2026-05-20T01:50:41+00:00,...,NaN,NaN,actual,NaN,3.940796,3.940796,NaN,0.788815,mispricing_abs,0.013514
2,5.0,AAPL,NaN,NaN,NaN,WARN,NaN,0.0,static_sp500_core_sample,2026-05-20T01:50:41+00:00,...,NaN,NaN,actual,NaN,3.928286,3.928286,NaN,0.771797,mispricing_abs,0.027027
3,6.0,ABBV,NaN,NaN,NaN,WARN,NaN,0.0,static_sp500_core_sample,2026-05-20T01:50:41+00:00,...,NaN,NaN,actual,NaN,3.915777,3.915777,NaN,0.754780,mispricing_abs,0.040541
4,7.0,AI.PA,NaN,France,NaN,WARN,NaN,0.0,static_euro_stoxx_50,2026-05-20T01:50:41+00:00,...,NaN,NaN,actual,NaN,3.903267,3.903267,NaN,0.737762,mispricing_abs,0.054054
5,8.0,AIR.PA,NaN,France,NaN,WARN,NaN,0.0,static_euro_stoxx_50,2026-05-20T01:50:41+00:00,...,NaN,NaN,actual,NaN,3.890757,3.890757,NaN,0.720744,mispricing_abs,0.067568
6,9.0,ALV.DE,NaN,Germany,NaN,WARN,NaN,0.0,static_euro_stoxx_50,2026-05-20T01:50:41+00:00,...,NaN,NaN,actual,NaN,3.878248,3.878248,NaN,0.703726,mispricing_abs,0.081081
7,10.0,AMP.MI,NaN,Italy,NaN,WARN,NaN,0.0,static_ftse_mib,2026-05-20T01:50:41+00:00,...,NaN,NaN,actual,NaN,3.865738,3.865738,NaN,0.686708,mispricing_abs,0.094595
8,11.0,AMZN,NaN,NaN,NaN,WARN,NaN,0.0,static_sp500_core_sample,2026-05-20T01:50:41+00:00,...,NaN,NaN,actual,NaN,3.853228,3.853228,NaN,0.669690,mispricing_abs,0.108108
9,12.0,ASML.AS,NaN,Netherlands,NaN,WARN,NaN,0.0,static_euro_stoxx_50,2026-05-20T01:50:41+00:00,...,NaN,NaN,actual,NaN,3.840718,3.840718,NaN,0.652672,mispricing_abs,0.121622
10,13.0,ATL.MI,NaN,Italy,NaN,WARN,NaN,0.0,static_ftse_mib,2026-05-20T01:50:41+00:00,...,NaN,NaN,actual,NaN,3.828209,3.828209,NaN,0.635654,mispricing_abs,0.135135


## 6. Quintile Portfolios


In [7]:
usable_forward_returns = (
    "forward_return" in signals.columns
    and pd.to_numeric(signals["forward_return"], errors="coerce").notna().sum() >= max(MIN_TICKERS, 10)
)
usable_relative_mispricing = (
    "mispricing_rel" in signals.columns
    and pd.to_numeric(signals["mispricing_rel"], errors="coerce").notna().sum() >= max(MIN_TICKERS, 10)
)
usable_absolute_mispricing = (
    "mispricing_abs" in signals.columns
    and pd.to_numeric(signals["mispricing_abs"], errors="coerce").notna().sum() >= max(MIN_TICKERS, 10)
)
if usable_forward_returns:
    return_col = "forward_return"
elif usable_relative_mispricing:
    return_col = "mispricing_rel"
elif usable_absolute_mispricing:
    return_col = "mispricing_abs"
else:
    return_col = "zscore"
    print("[WARN] No return/mispricing column has enough coverage; using zscore as last-resort quintile diagnostic.")
if return_col != "forward_return":
    print(f"[WARN] forward_return unavailable or too sparse; using {return_col} for quintile diagnostics fallback.")

if signals.empty or "zscore" not in signals.columns or return_col not in signals.columns:
    quintile_returns = pd.DataFrame(columns=["date", "quantile", "return", "name_count"])
else:
    quintile_returns = make_quantile_portfolios(signals, "zscore", return_col=return_col, q=5)

experiment_status = build_status_from_signals(signals, quintile_returns)
save_status_file(experiment_status, OUTPUTROOT)
if experiment_status.status != "ok":
    print(f"[WARN] Experiment status: {experiment_status.status} ({experiment_status.reason})")

risk_report = quintile_risk_report(quintile_returns, periods_per_year=12)
risk_report_path = OUTPUTROOT / "MLStockLab_quintile_risk.csv"
risk_report.to_csv(risk_report_path, index=True)
quintile_returns.to_csv(TABLESDIR / "MLStockLab_quintile_returns.csv", index=False)
quintile_metrics = risk_report.reset_index()
quintile_metrics["return_col"] = return_col
quintile_metrics.to_csv(TABLESDIR / "MLStockLab_quintile_metrics.csv", index=False)
display(risk_report)


[WARN] forward_return unavailable or too sparse; using mispricing_abs for quintile diagnostics fallback.


,mean_return,annualized_vol,sharpe,max_drawdown,avg_n_names
portfolio,,,,,
Q1,2.507135,0.0,NaN,0.0,15.0
Q2,3.302801,0.0,NaN,0.0,15.0
Q3,3.484191,0.0,NaN,0.0,14.0
Q4,3.665582,0.0,NaN,0.0,15.0
Q5,3.853228,0.0,NaN,0.0,15.0
LS,1.346093,0.0,NaN,0.0,30.0


## 7. ML Model Comparison


In [8]:
comparison = []
if X.empty or y.empty:
    comparison.append({"model": MODEL, "fair_value_corr": pd.NA, "mispricing_std": pd.NA, "note": "empty_model_dataset"})
else:
    for model_name in ["ols", "lasso", "rf", "gbrt", "ensemble"]:
        try:
            pred = PeerImpliedValuator(model=model_name).fit_predict(X, y)
            mp = compute_relative_mispricing(pred, y)
            comparison.append({"model": model_name, "fair_value_corr": pred.corr(y), "mispricing_std": mp.std()})
        except Exception as exc:
            comparison.append({"model": model_name, "fair_value_corr": pd.NA, "mispricing_std": pd.NA, "error": str(exc)})
comparison_df = pd.DataFrame(comparison)
comparison_df.to_csv(TABLESDIR / "MLStockLab_model_comparison.csv", index=False)
display(comparison_df)


,model,fair_value_corr,mispricing_std
0,ols,0.025247,NaN
1,lasso,0.025247,NaN
2,rf,0.701724,NaN
3,gbrt,1.000000,NaN
4,ensemble,0.962330,NaN


## 8. Expected Return Prediction


In [9]:
if signals.empty:
    prediction_metrics = pd.DataFrame([{
        "status": "skipped",
        "reason": "signals_empty",
        "r2_os": pd.NA,
        "train_rows": 0,
        "test_rows": 0,
    }])
elif "forward_return" in signals.columns and signals["forward_return"].notna().sum() > 10 and feature_cols:
    train_idx, test_idx = temporal_train_test_split(signals)
    split_meta = describe_temporal_split(signals, train_idx, test_idx)
    yret = pd.to_numeric(signals["forward_return"], errors="coerce")
    row = {**split_meta, "status": "skipped", "reason": "", "r2_os": pd.NA}
    if len(train_idx) > 5 and len(test_idx) > 0:
        try:
            predictor = ExpectedReturnModel(model="rf")
            predictor.fit(signals.loc[train_idx, feature_cols], yret.loc[train_idx])
            pred = predictor.predict(signals.loc[test_idx, feature_cols])
            row["status"] = "ok"
            row["r2_os"] = oos_r2(yret.loc[test_idx], pred)
        except Exception as exc:
            row["reason"] = f"prediction_failed:{type(exc).__name__}"
    else:
        row["reason"] = "temporal_split_too_small"
    prediction_metrics = pd.DataFrame([row])
else:
    prediction_metrics = pd.DataFrame([{
        "status": "skipped",
        "reason": "forward_return unavailable or too sparse",
        "r2_os": pd.NA,
        "train_rows": 0,
        "test_rows": 0,
    }])
prediction_metrics.to_csv(TABLESDIR / "MLStockLab_prediction_metrics.csv", index=False)
display(prediction_metrics)


,status,reason,r2_os,train_rows,test_rows
0,skipped,forward_return unavailable or too sparse,<NA>,0,0


## 9. Visuals


In [10]:
# 9. Fintech Visual Dashboard
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if signals.empty:
    print("No ML signals available yet. Run the previous cells or the one-shot artifact runner.")
else:
    color_col = "zscore" if "zscore" in signals.columns else None
    if {"market_value", "fair_value_hat"}.issubset(signals.columns):
        fig = px.scatter(
            signals,
            x="market_value",
            y="fair_value_hat",
            color=color_col,
            hover_name="ticker" if "ticker" in signals.columns else None,
            title="Fair Value vs Market Value",
            template="plotly_white",
            color_continuous_scale="RdYlGn",
        )
        max_axis = max(float(signals["market_value"].max()), float(signals["fair_value_hat"].max())) if len(signals) else 1
        fig.add_trace(go.Scatter(x=[0, max_axis], y=[0, max_axis], mode="lines", name="Fair = Market", line=dict(color="#667085", dash="dash")))
        fig.update_layout(height=540, margin=dict(l=20, r=20, t=60, b=20))
        fig.write_html(FIGURESDIR / "MLStockLab_fair_value_vs_market.html")
        fig.show()

    if "zscore" in signals.columns:
        fig2 = px.histogram(signals, x="zscore", nbins=30, title="Mispricing z-score distribution", template="plotly_white", color_discrete_sequence=["#01696f"])
        fig2.add_vline(x=0, line_dash="dash", line_color="#667085")
        fig2.write_html(FIGURESDIR / "MLStockLab_zscore_distribution.html")
        fig2.show()

    rank_col = "zscore" if "zscore" in signals.columns else "mispricing_rel" if "mispricing_rel" in signals.columns else None
    if rank_col and "ticker" in signals.columns:
        top_bottom = pd.concat([
            signals.sort_values(rank_col, ascending=False).head(15).assign(bucket="Top"),
            signals.sort_values(rank_col, ascending=True).head(15).assign(bucket="Bottom"),
        ])
        fig3 = px.bar(top_bottom, x="ticker", y=rank_col, color="bucket", title=f"Top / Bottom names by {rank_col}", template="plotly_white", color_discrete_map={"Top":"#01696f", "Bottom":"#da7101"})
        fig3.update_layout(height=480, margin=dict(l=20, r=20, t=60, b=20))
        fig3.write_html(FIGURESDIR / "MLStockLab_top_bottom_scores.html")
        fig3.show()

    if not quintile_returns.empty and {"quantile", "return"}.issubset(quintile_returns.columns):
        fig4 = px.bar(quintile_returns, x="quantile", y="return", color="quantile", title="Quintile / Long-Short return diagnostic", template="plotly_white")
        fig4.write_html(FIGURESDIR / "MLStockLab_quintile_returns.html")
        fig4.show()


## 10. One-shot Artifact Runner


In [11]:
ls_metrics = risk_report.loc["LS"].to_dict() if "risk_report" in globals() and "LS" in risk_report.index else {}
pred_metrics = prediction_metrics.iloc[0].to_dict() if "prediction_metrics" in globals() and not prediction_metrics.empty else {}
metrics = pd.DataFrame([{
    "status": experiment_status.status if "experiment_status" in globals() else "unknown",
    "reason": experiment_status.reason if "experiment_status" in globals() else None,
    "model": MODEL,
    "target": ML_LAB_CONFIG.get("target", "market_value"),
    "feature_blocks": ",".join(ML_LAB_CONFIG.get("feature_blocks", [])),
    "universe": ML_LAB_CONFIG.get("universe", ""),
    "max_rows": MAX_ROWS,
    "min_dates": MIN_DATES,
    "min_tickers": MIN_TICKERS,
    "panel_rows": len(panel),
    "signal_rows": len(signals),
    "quintile_return_col": globals().get("return_col", pd.NA),
    "r2_os": pred_metrics.get("r2_os", pd.NA),
    "split_criterion": pred_metrics.get("split_criterion", pd.NA),
    "train_end_date": pred_metrics.get("train_end_date", pd.NA),
    "test_start_date": pred_metrics.get("test_start_date", pd.NA),
    "sharpe_ls": ls_metrics.get("sharpe", pd.NA),
    "annualized_vol_ls": ls_metrics.get("annualized_vol", pd.NA),
    "avg_n_names_ls": ls_metrics.get("avg_n_names", pd.NA),
}])
metrics.to_csv(TABLESDIR / "MLStockLab_metrics.csv", index=False)
display(metrics)


,status,reason,model,target,feature_blocks,universe,max_rows,min_dates,min_tickers,panel_rows,signal_rows,quintile_return_col,r2_os,split_criterion,train_end_date,test_start_date,sharpe_ls,annualized_vol_ls,avg_n_names_ls
0,ok,None,ols,market_value,"value,quality,momentum",All Available,2000,2,5,151,75,mispricing_abs,None,<NA>,<NA>,<NA>,NaN,0.0,30.0


## 11. Caveats


- Use time-aware splits for predictive claims.
- Cross-sectional fair value is not a DCF replacement.
- Quintile backtests are research diagnostics and need transaction costs, liquidity and factor controls before production use.
- Missing Database Finanziario coverage is surfaced as empty artifacts, not silently imputed.


## 12. Export Summary


In [12]:
exports = sorted([str(p.relative_to(OUTPUTROOT)) for p in OUTPUTROOT.rglob("*") if p.is_file()])
for item in exports:
    print(item)


MLStockLab_experiment_summary.csv
MLStockLab_experiment_summary.json
MLStockLab_quintile_risk.csv
MLStockLab_status.json
ML_Stock_Lab_Experiments_TESTED.ipynb
figures/MLStockLab_fair_value_vs_market.html
figures/MLStockLab_quintile_returns.html
figures/MLStockLab_top_bottom_scores.html
figures/MLStockLab_zscore_distribution.html
tables/MLStockLab_bottom.csv
tables/MLStockLab_metrics.csv
tables/MLStockLab_model_comparison.csv
tables/MLStockLab_panel.csv
tables/MLStockLab_prediction_metrics.csv
tables/MLStockLab_quintile_metrics.csv
tables/MLStockLab_quintile_returns.csv
tables/MLStockLab_signals.csv
tables/MLStockLab_top.csv


## 13. Integration Notes


The same `MLStockLab_*` artifacts are consumed by the Streamlit ML Lab page and can be generated by the orchestration job `ml_stock_lab_experiments_refresh`.


## 14. Summary


In [13]:
ls_metrics = risk_report.loc["LS"].to_dict() if "risk_report" in globals() and "LS" in risk_report.index else {}
pred_metrics = prediction_metrics.iloc[0].to_dict() if "prediction_metrics" in globals() and not prediction_metrics.empty else {}
hyperparams = {
    "target": ML_LAB_CONFIG.get("target", "market_value"),
    "feature_blocks": ",".join(ML_LAB_CONFIG.get("feature_blocks", [])),
    "model": MODEL,
    "max_rows": MAX_ROWS,
    "universe": ML_LAB_CONFIG.get("universe", ""),
    "tickers": ",".join(ML_LAB_CONFIG.get("tickers", [])),
    "currency": ML_LAB_CONFIG.get("currency", ""),
    "fx_pair": ML_LAB_CONFIG.get("fx_pair", ""),
    "min_dates": MIN_DATES,
    "min_tickers": MIN_TICKERS,
    "quintile_return_col": globals().get("return_col", None),
}
summary = build_experiment_summary(
    hyperparams=hyperparams,
    panel_validation=panel_val,
    metrics={
        "panel_rows": len(panel),
        "signal_rows": len(signals),
        "feature_count": len(feature_cols),
        "r2_os": pred_metrics.get("r2_os", pd.NA),
        "train_end_date": pred_metrics.get("train_end_date", pd.NA),
        "test_start_date": pred_metrics.get("test_start_date", pd.NA),
        "split_criterion": pred_metrics.get("split_criterion", pd.NA),
        "sharpe_ls": ls_metrics.get("sharpe", pd.NA),
        "annualized_vol_ls": ls_metrics.get("annualized_vol", pd.NA),
        "avg_n_names_ls": ls_metrics.get("avg_n_names", pd.NA),
    },
    status=experiment_status,
)
summary_path = save_experiment_summary(summary, OUTPUTROOT)
print("Experiment summary saved:", summary_path)
summary


Experiment summary saved: /Users/itsgennymac/GitHub/machine-learning-for-trading/research_platform_definitive/output/ml_stock_lab/MLStockLab_experiment_summary.json


{'param_target': 'market_value',
 'param_feature_blocks': 'value,quality,momentum',
 'param_model': 'ols',
 'param_max_rows': 2000,
 'param_universe': 'All Available',
 'param_tickers': '',
 'param_currency': 'USD',
 'param_fx_pair': 'DX-Y.NYB',
 'param_min_dates': 2,
 'param_min_tickers': 5,
 'param_quintile_return_col': 'mispricing_abs',
 'panel_ok': True,
 'panel_min_date': '2024-03-31',
 'panel_max_date': '2026-05-21',
 'panel_n_dates': 3,
 'panel_n_tickers': 76,
 'panel_reason': None,
 'metric_panel_rows': 151,
 'metric_signal_rows': 75,
 'metric_feature_count': 5,
 'metric_r2_os': None,
 'metric_train_end_date': <NA>,
 'metric_test_start_date': <NA>,
 'metric_split_criterion': <NA>,
 'metric_sharpe_ls': nan,
 'metric_annualized_vol_ls': 0.0,
 'metric_avg_n_names_ls': 30.0,
 'status_status': 'ok',
 'status_reason': None,
 'status_details': {'signal_rows': 75, 'quintile_return_rows': 6}}